<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 4 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Joining lake tables and internal tables</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Query orders in the lake, join the Doris customer table, and verify the loaded results.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Target Doris 4.1.3 · Order data · Isolated lab database</span>
</div>

This lab prepares ten orders in an Iceberg lake table, joins the Doris internal customer table, and verifies the loaded results: ten rows and a pre-tax amount of 12220.60.

[Course notes](course4_querying_external_data.md) · [Course home](../README.md)


## Prepare the lake table lab

Complete Lab 1 first. Running the next cell connects to the course Doris instance, starts two course-specific auxiliary containers for MinIO object storage and Iceberg REST Catalog, and creates a ten-order sample in the lake. The first startup requires downloading images; subsequent runs reuse the data.

Doris continues to use the original single container. The auxiliary services listen on local ports 51900 and 51818 and use public local-lab credentials. Confirm that the ports are free before running; see the [lake table environment notes](../../environments/lakehouse/README.md) for shutdown and data retention. This lab rebuilds only the internal tables customers_sample and orders_from_lake; lake tables are stored separately for each lab database.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.lakehouse import prepare_lakehouse
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows, sample
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()
source = prepare_lakehouse(lab, start=True)




## 1. Query the lake table directly

Doris locates the Iceberg table through a Catalog, then reads its data files. The next cell checks the table type, query plan, and order contents. Expect ten orders and a pre-tax amount of 12220.60; at this point, the orders are still stored in the lake table.


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10,"12220.60")])
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1");
lab.sql(f"SELECT order_id, customer_id, order_date, order_amount FROM {source} ORDER BY order_id", title="Ten orders in the lake")


## 2. Join the internal customer table

This step rebuilds only customers_sample and orders_from_lake. First create a unique-key customer table and join it to the lake orders on customer_id: each order should match exactly one customer record.

Then load the six order fields into orders_from_lake and compare the lake table and internal table field by field. After joining and loading, there should still be ten orders and a pre-tax amount of 12220.60.


In [ ]:
lab.execute("DROP TABLE IF EXISTS customers_sample")
lab.execute('CREATE TABLE customers_sample (customer_id BIGINT, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("customers_sample", ["customer_id", "customer_name"],
           [(r["customer_id"], r["customer_name"]) for r in sample()["customers"]])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN customers_sample c ON o.customer_id=c.customer_id"),
       [(10, "12220.60")])
lab.execute("DROP TABLE IF EXISTS orders_from_lake")
ddl = order_ddl("orders_from_lake")
show_sql("CREATE TABLE SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_from_lake ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_lake ORDER BY order_id"),
       history_rows())
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_from_lake ORDER BY order_id", title="Order details after loading")


## Completion and boundaries

You can query the Iceberg orders table through a Catalog; order row counts and amounts remain unchanged after joining internal customers; the loaded internal table matches the lake table field by field.


## Your turn

Compare execution plans for directly querying the lake table and querying orders_from_lake, and identify the scan target in each. Explain the role of Iceberg table metadata in queries and why the customer dimension table must be unique on customer_id.


## Independent exercise

Use LEFT JOIN to query the order count, amount, and missing-customer count for the second day in the lake table, then compare the scan plans for the lake table and internal table. Expect 5 orders, 8276.40, and 0 missing customers.

Write and run your code in the next cell, then expand the reference solution after finishing. A blank exercise will not be automatically marked complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
query = f"""SELECT COUNT(*) AS orders, SUM(o.order_amount) AS amount,
SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers
FROM {source} o LEFT JOIN customers_sample c ON o.customer_id=c.customer_id
WHERE o.order_date='2013-01-02'"""
lab.sql(query, title="Customer matches for the second day's lake orders")
expect(lab.query(query), [(5,"8276.40",0)])
lab.sql("EXPLAIN SELECT * FROM orders_from_lake WHERE order_date='2013-01-02'", title="Internal table scan")
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_date='2013-01-02'", title="Lake table scan")
```

</details>
